In [1]:
# ============================================================
# 03_nlp_preprocessing.py — Phase C: NLP Preprocessing
# Internship Project: NLP Sentiment Analysis (IMDB 50K)
# ============================================================

import pandas as pd
import numpy as np
import nltk, os, re, time
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# ── NLTK downloads ────────────────────────────────────────────
def download_nltk():
    for pkg in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'omw-1.4']:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

# ── Config ───────────────────────────────────────────────────
DATA_PATH = "data/imdb_cleaned.csv"
OUT_PATH  = "data/imdb_preprocessed.csv"

# ── Preprocessing class ───────────────────────────────────────
class NLPPreprocessor:
    def __init__(self):
        download_nltk()
        self.lemmatizer  = WordNetLemmatizer()
        self.stop_words  = set(stopwords.words('english'))
        # keep negations — critical for sentiment
        negations = {'no','not','nor','never','neither','nothing','nowhere',
                     'nobody','cannot','can\'t','won\'t','don\'t','isn\'t',
                     'aren\'t','wasn\'t','weren\'t','hasn\'t','hadn\'t',
                     'didn\'t','doesn\'t','wouldn\'t','couldn\'t','shouldn\'t'}
        self.stop_words -= negations

    def tokenize(self, text: str) -> list:
        """Simple whitespace tokenizer (fast, no NLTK dependency at runtime)."""
        return text.split()

    def remove_stopwords(self, tokens: list) -> list:
        return [t for t in tokens if t not in self.stop_words]

    def lemmatize(self, tokens: list) -> list:
        return [self.lemmatizer.lemmatize(t) for t in tokens]

    def preprocess(self, text: str) -> str:
        """Full NLP pipeline: tokenize → remove stops → lemmatize → join."""
        tokens   = self.tokenize(text)
        tokens   = self.remove_stopwords(tokens)
        tokens   = self.lemmatize(tokens)
        return ' '.join(tokens)

    def batch_process(self, series: pd.Series, sample_log: int = 10_000) -> pd.Series:
        total   = len(series)
        results = []
        t0      = time.time()
        for i, text in enumerate(series):
            results.append(self.preprocess(text))
            if (i + 1) % sample_log == 0:
                elapsed = time.time() - t0
                pct     = (i+1) / total * 100
                eta     = elapsed / (i+1) * (total - i - 1)
                print(f"  {i+1:>6}/{total:,}  ({pct:5.1f}%)  elapsed {elapsed:.0f}s  ETA {eta:.0f}s")
        return pd.Series(results, index=series.index)

def main():
    print("=" * 55)
    print("  Phase C — NLP Preprocessing")
    print("=" * 55)

    print("\n[1/3] Loading cleaned data …")
    df = pd.read_csv(DATA_PATH)
    print(f"      {len(df):,} reviews loaded")

    proc = NLPPreprocessor()
    print(f"\n[2/3] Running NLP pipeline (tokenize → stopword removal → lemmatize) …")
    print(f"      Stop words loaded : {len(proc.stop_words):,}")
    print(f"      Negations kept    : not, no, never, can't, won't, …\n")

    df['processed_review'] = proc.batch_process(df['clean_review'])

    # ── Stats ──────────────────────────────────────────────
    df['proc_len']  = df['processed_review'].apply(lambda x: len(x.split()))
    df['clean_len'] = df['clean_review'].apply(lambda x: len(x.split()))
    reduction = (1 - df['proc_len'].mean() / df['clean_len'].mean()) * 100

    print(f"\n[3/3] Saving preprocessed data → {OUT_PATH}")
    os.makedirs("data", exist_ok=True)
    df.to_csv(OUT_PATH, index=False)

    print("\n" + "="*55)
    print("  Preprocessing Summary")
    print("="*55)
    print(f"  Avg words (before) : {df['clean_len'].mean():.1f}")
    print(f"  Avg words (after)  : {df['proc_len'].mean():.1f}")
    print(f"  Vocabulary reduction: {reduction:.1f}%")
    print(f"\n  Before: '{df['clean_review'].iloc[0][:80]}…'")
    print(f"  After : '{df['processed_review'].iloc[0][:80]}…'")
    print(f"\n✅ Phase C complete! {len(df):,} reviews preprocessed.")

if __name__ == "__main__":
    main()


  Phase C — NLP Preprocessing

[1/3] Loading cleaned data …
      50,000 reviews loaded

[2/3] Running NLP pipeline (tokenize → stopword removal → lemmatize) …
      Stop words loaded : 182
      Negations kept    : not, no, never, can't, won't, …

   10000/50,000  ( 20.0%)  elapsed 9s  ETA 38s
   20000/50,000  ( 40.0%)  elapsed 15s  ETA 23s
   30000/50,000  ( 60.0%)  elapsed 21s  ETA 14s
   40000/50,000  ( 80.0%)  elapsed 26s  ETA 7s
   50000/50,000  (100.0%)  elapsed 32s  ETA 0s

[3/3] Saving preprocessed data → data/imdb_preprocessed.csv

  Preprocessing Summary
  Avg words (before) : 234.2
  Avg words (after)  : 119.9
  Vocabulary reduction: 48.8%

  Before: 'one of the other reviewers has mentioned that after watching just oz episode you…'
  After : 'one reviewer mentioned watching oz episode hooked right exactly happened first t…'

✅ Phase C complete! 50,000 reviews preprocessed.
